In [1]:
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "dbrepo"], 
               capture_output=True)

from dbrepo.RestClient import RestClient
import pandas as pd
import numpy as np
import os
import time

print("All libraries imported successfully")

All libraries imported successfully


In [2]:

os.environ["DBREPO_PASSWORD"] = input("Enter your DBRepo password: ")

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="12534814",
    password=os.environ.get("DBREPO_PASSWORD")
)

DATABASE_ID = "13457a52-37f9-48d4-a078-6865e8d35981"
db = client.get_database(database_id=DATABASE_ID)
print("Connected to database:", db.name)

Connected to database: lake_water_quality


In [3]:
import os

# Set working directory
os.chdir(os.path.dirname(os.path.abspath("T2_1_schema_fixed.ipynb")))

# Load CSV
df = pd.read_csv("Lakes_Monitoring.csv")
print("CSV loaded:", df.shape)

# Drop system columns
df_clean = df.drop(columns=['_type', '_id', '_revision', '_page.next'])
print("Clean shape:", df_clean.shape)

CSV loaded: (1935, 116)
Clean shape: (1935, 112)


In [25]:

#fixing detection limit values
cols_to_fix = [
    'suspend_medziagos', 'sarmingumas', 'skaidrumas',
    'biochem_deg_suvartojimas', 'amonio_azotas', 'nitritu_azotas',
    'nitratu_azotas', 'fosfatu_fosforas', 'fosforas_bendras',
    'chlorofilas_a', 'gyvsidabris', 'kadmis', 'nikelis', 'svinas',
    'varis', 'chromas', 'vanadis', 'aliuminis', 'alavas', 'arsenas',
    'cinkas', 'antracenas', 'fluorantenas', 'naftalenas',
    'benz_a_pirenas', 'benz_b_fluorantenas', 'benz_k_fluorantenas',
    'benz_ghi_perilenas', 'inden_123_cd_pirenas', 'p4_n_nonilfenolis',
    'p4_n_oktilfenolis', 'p4_nonilfenolis_sakot', 'p4_tert_oktilfenolis',
    'nonilfenoliai', 'pentachlorfenolis', 'benzenas', 'p12_dichloretanas',
    'p123_trichlorbenzenas', 'p124_trichlorbenzenas', 'heksachlorbutadienas',
    'trichloretilenas', 'tetrachlormetanas', 'dichlormetanas',
    'tetrachloretilenas', 'trichlormetanas', 'aldrinas', 'dieldrinas',
    'izodrinas', 'endrinas', 'alfa_heksachlorcikloheksanas',
    'beta_heksachlorcikloheksanas', 'gama_heksachlorcikloheksanas',
    'heksachlorbenzenas', 'pentachlorbenzenas', 'alfa_endosulfanas',
    'beta_endosulfanas', 'o_p_ddt', 'p_p_ddd', 'p_p_ddt', 'p_p_dde',
    'simazinas', 'atrazinas', 'diuronas', 'izoproturonas', 'chinoksifenas',
    'aklonifenas', 'cibutrinas', 'terbutrinas', 'chlorpyrifosas',
    'chlorfenvinfosas', 'trifluralinas', 'heptachloras',
    'heptachloro_epoksidas', 'tributilalavo_katijonas',
    'di2_etilheksilftalatas', 'bde_28', 'bde_47', 'bde_85', 'bde_99',
    'bde_100', 'bde_153', 'bde_154', 'pcb_28', 'pcb_52', 'pcb_101',
    'pcb_118', 'pcb_138', 'pcb_153', 'pcb_180', 'pfos', 'dikofolis',
    'alachloras', 'bifenoksas', 'cipermetrinas', 'dichlorvosas',
    'p4_t_oktilfenolio_dietoksilatas', 'p4_t_oktilfenolio_monoetoksilatas',
    'p4_t_oktilfenolio_trietoksilatas'
]

def replace_below_detection(value):
    if value is None:
        return np.nan
    if isinstance(value, float):
        return value
    val_str = str(value).strip()
    if val_str in ['', '-', 'nan', 'None', 'Nematuota', 'nematuota']:
        return np.nan
    if val_str.startswith('<'):
        num_str = val_str.replace('<', '').replace(',', '.').strip()
        try:
            return float(num_str) / 2
        except:
            return np.nan
    try:
        return float(val_str.replace(',', '.'))
    except:
        return np.nan

for col in cols_to_fix:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(replace_below_detection)

print("Fix applied!")
print("Sample:", df_clean['nitritu_azotas'].head(3).tolist())
print("dtype:", df_clean['nitritu_azotas'].dtype)

Fix applied!
Sample: [0.0025, 0.002, 0.003]
dtype: float64


In [26]:
# --- lake ---
df_lake = df_clean[['telkinio_pav', 'regionas']].drop_duplicates()
df_lake.columns = ['lake_name', 'region']
df_lake = df_lake.reset_index(drop=True)
df_lake.index.name = "lake_id"
print("Lakes:", len(df_lake))

# --- sampling_station ---
df_lake_lookup = df_lake.reset_index()
df_station = df_clean[['m_vietos_kodas', 'm_vietos_pav',
                        'koord', 'telkinio_pav']].copy()
df_station = df_station.drop_duplicates(subset=['m_vietos_kodas'])
df_station.columns = ['station_code', 'station_name',
                      'coordinates', 'lake_name']
df_station = df_station.merge(
    df_lake_lookup[['lake_id', 'lake_name']],
    on='lake_name', how='left'
)
df_station = df_station.drop(columns=['lake_name'])
df_station['lake_id'] = df_station['lake_id'].fillna(0).astype(int)
df_station = df_station.reset_index(drop=True)
df_station.index.name = "station_id"
print("Stations:", len(df_station))

# --- sampling_event ---
df_event = df_clean[['m_vietos_kodas', 'data']].copy()
df_event.columns = ['station_code', 'sampled_on']
df_event = df_event.reset_index(drop=True)
df_event.index.name = "event_id"
print("Events:", len(df_event))

# --- water_quality_measurement ---
measurement_cols = [
    'vandens_temp', 'suspend_medziagos', 'sarmingumas',
    'deguonis_istirpes', 'ph', 'skaidrumas', 'elektr_laidis',
    'biochem_deg_suvartojimas', 'amonio_azotas', 'nitritu_azotas',
    'nitratu_azotas', 'azotas_mineralinis', 'azotas_bendras',
    'fosfatu_fosforas', 'fosforas_bendras', 'anglingumas',
    'chlorofilas_a', 'kalcio_karbonatas', 'gyvsidabris', 'kadmis',
    'nikelis', 'svinas', 'varis', 'chromas', 'vanadis', 'aliuminis',
    'alavas', 'arsenas', 'cinkas', 'antracenas', 'fluorantenas',
    'naftalenas', 'benz_a_pirenas', 'benz_b_fluorantenas',
    'benz_k_fluorantenas', 'benz_ghi_perilenas', 'inden_123_cd_pirenas',
    'p4_n_nonilfenolis', 'p4_n_oktilfenolis', 'p4_nonilfenolis_sakot',
    'p4_tert_oktilfenolis', 'nonilfenoliai', 'pentachlorfenolis',
    'benzenas', 'p12_dichloretanas', 'p123_trichlorbenzenas',
    'p124_trichlorbenzenas', 'heksachlorbutadienas', 'trichloretilenas',
    'tetrachlormetanas', 'dichlormetanas', 'tetrachloretilenas',
    'trichlormetanas', 'aldrinas', 'dieldrinas', 'izodrinas', 'endrinas',
    'alfa_heksachlorcikloheksanas', 'beta_heksachlorcikloheksanas',
    'gama_heksachlorcikloheksanas', 'heksachlorbenzenas',
    'pentachlorbenzenas', 'alfa_endosulfanas', 'beta_endosulfanas',
    'o_p_ddt', 'p_p_ddd', 'p_p_ddt', 'p_p_dde', 'simazinas',
    'atrazinas', 'diuronas', 'izoproturonas', 'chinoksifenas',
    'aklonifenas', 'cibutrinas', 'terbutrinas', 'chlorpyrifosas',
    'chlorfenvinfosas', 'trifluralinas', 'heptachloras',
    'heptachloro_epoksidas', 'tributilalavo_katijonas',
    'di2_etilheksilftalatas', 'bde_28', 'bde_47', 'bde_85', 'bde_99',
    'bde_100', 'bde_153', 'bde_154', 'pcb_28', 'pcb_52', 'pcb_101',
    'pcb_118', 'pcb_138', 'pcb_153', 'pcb_180', 'pfos', 'dikofolis',
    'alachloras', 'bifenoksas', 'cipermetrinas', 'dichlorvosas',
    'p4_t_oktilfenolio_dietoksilatas', 'p4_t_oktilfenolio_monoetoksilatas',
    'p4_t_oktilfenolio_trietoksilatas'
]
df_measurement = df_clean[measurement_cols].copy()
df_measurement.index.name = "measurement_id"
print("Measurements:", len(df_measurement))
print()
print("All dataframes ready!")

Lakes: 346
Stations: 386
Events: 1935
Measurements: 1935

All dataframes ready!


In [28]:
print("Creating lake table...")
time.sleep(5)

table_lake = client.create_table(
    database_id=DATABASE_ID,
    name="lake",
    is_public=True,
    is_schema_public=True,
    dataframe=df_lake,
    description="Unique lakes with their administrative region in Lithuania",
    with_data=True
)
print("lake table created! Rows:", len(df_lake))
print("Table ID:", table_lake.id)
time.sleep(10)
print("Ready for next table")

Creating lake table...
2026-05-28 16:09:15,385 root         WARNING default to 'text' for column lake_name and type <class 'numpy.dtype'>
2026-05-28 16:09:15,386 root         WARNING default to 'text' for column region and type <class 'numpy.dtype'>
lake table created! Rows: 346
Table ID: 5ab83a36-4f2a-4363-81c0-99eb6fd0d82e
Ready for next table


In [33]:
print("Creating sampling_station table...")

table_station = client.create_table(
    database_id=DATABASE_ID,
    name="sampling_station",
    is_public=True,
    is_schema_public=True,
    dataframe=df_station,
    description="Monitoring stations with official code, coordinates and lake_id FK",
    with_data=True
)
print("sampling_station table created! Rows:", len(df_station))
print("Table ID:", table_station.id)
time.sleep(10)
print("Ready for next table")

Creating sampling_station table...
2026-05-28 16:10:16,509 root         WARNING default to 'text' for column station_code and type <class 'numpy.dtype'>
2026-05-28 16:10:16,510 root         WARNING default to 'text' for column station_name and type <class 'numpy.dtype'>
2026-05-28 16:10:16,512 root         WARNING default to 'text' for column coordinates and type <class 'numpy.dtype'>
sampling_station table created! Rows: 386
Table ID: ef0031db-e1cf-43d9-a55d-e7e44ecc687b
Ready for next table


In [34]:
# create sampling_event with only station_code and sampled_on
df_event= df_clean[['m_vietos_kodas', 'data']].copy()
df_event.columns = ['station_code', 'sampled_on']
df_event = df_event.reset_index(drop=True)
df_event.index.name = "event_id"

print("Events:", len(df_event))
print(df_event.head())

# Create table with data
table_event = client.create_table(
    database_id=DATABASE_ID,
    name="sampling_event",
    is_public=True,
    is_schema_public=True,
    dataframe=df_event,
    description="Sampling visits - one row per station per date",
    with_data=False
)
print("Table created:", table_event.id)
time.sleep(15)

# Import in small chunks
chunk_size = 50
total = len(df_event)
success_count = 0

for start in range(0, total, chunk_size):
    end = min(start + chunk_size, total)
    chunk = df_event.iloc[start:end].reset_index()
    
    for attempt in range(3):
        try:
            client.import_table_data(
                database_id=DATABASE_ID,
                table_id=table_event.id,
                dataframe=chunk
            )
            success_count += end - start
            print(f"Rows {start}-{end} imported ({success_count}/{total})")
            time.sleep(8)
            break
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(15)

print(f"Done! {success_count}/{total} rows")

Events: 1935
         station_code  sampled_on
event_id                         
0              LTL116  2021-07-29
1               LTL68  2021-07-28
2              LTL536  2018-08-20
3              LTL338  2021-07-22
4              LTL344  2020-07-27
2026-05-28 16:10:32,993 root         WARNING default to 'text' for column station_code and type <class 'numpy.dtype'>
2026-05-28 16:10:32,995 root         WARNING default to 'text' for column sampled_on and type <class 'numpy.dtype'>
Table created: 3c8ea8b6-57fb-4cdf-92b0-7fa2b2e114ae
Rows 0-50 imported (50/1935)
Attempt 1 failed: Failed to insert table data: data service failed to establish connection to metadata service
Rows 50-100 imported (100/1935)
Attempt 1 failed: Failed to import table data: {"status":"BAD_REQUEST","message":"Failed to import tuple: (conn=34760) Unknown column 'sampling_event.station_code' in 'field list': (conn=34760) Unknown column 'sampling_event.station_code' in 'field list'","code":"error.request.invalid"}
Att

In [41]:
# Add event_id to link with sampling_event
df_measurement = df_clean[measurement_cols].copy()
df_measurement['event_id'] = range(len(df_measurement))
df_measurement.index.name = "measurement_id"
print("Measurements:", len(df_measurement))
print("event_id added:", 'event_id' in df_measurement.columns)
print(df_measurement[['event_id']].head())

# Create table structure
print("Creating water_quality_measurement table...")
table_meas = client.create_table(
    database_id=DATABASE_ID,
    name="water_quality_measurement",
    is_public=True,
    is_schema_public=True,
    dataframe=df_measurement,
    description="All physicochemical and pollutant measurements for each sampling event",
    with_data=False
)
print("Structure created! Table ID:", table_meas.id)
time.sleep(15)
print("Now importing data in chunks...")

# Import in chunks of 100
chunk_size = 100
total = len(df_measurement)
success_count = 0

for start in range(0, total, chunk_size):
    end = min(start + chunk_size, total)
    chunk = df_measurement.iloc[start:end].reset_index()
    
    for attempt in range(3):
        try:
            client.import_table_data(
                database_id=DATABASE_ID,
                table_id=table_meas.id,
                dataframe=chunk
            )
            success_count += end - start
            print(f"Rows {start}-{end} imported ({success_count}/{total})")
            time.sleep(3)
            break
        except Exception as e:
            print(f"Attempt {attempt+1} failed for rows {start}-{end}: {e}")
            time.sleep(10)

print(f"Done! Successfully imported {success_count}/{total} rows")
print("New Table ID:", table_meas.id)

Measurements: 1935
event_id added: True
                event_id
measurement_id          
0                      0
1                      1
2                      2
3                      3
4                      4
Creating water_quality_measurement table...
Structure created! Table ID: 0b680301-8907-4cad-af74-a345489adf5c
Now importing data in chunks...
Attempt 1 failed for rows 0-100: Failed to import table data: {"status":"BAD_REQUEST","message":"Failed to write dataset: schema malformed: Job aborted due to stage failure: Task 0 in stage 1356.0 failed 1 times, most recent failure: Lost task 0.0 in stage 1356.0 (TID 1362) (data-service-c464f4856-cj95g executor driver): java.sql.BatchUpdateException: (conn=1614894) Unknown column 'event_id' in 'field list'\n\tat org.mariadb.jdbc.export.ExceptionFactory.createBatchUpdate(ExceptionFactory.java:181)\n\tat org.mariadb.jdbc.BasePreparedStatement.executeBatchBulk(BasePreparedStatement.java:1805)\n\tat org.mariadb.jdbc.ClientPreparedStatemen